In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pandas as pd
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Set up headless browser
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

base_url = "https://www.bettingpros.com"
main_url = f"{base_url}/nfl/odds/player-futures/"
driver.get(main_url)
time.sleep(5)  # let JS load

# Get player links
player_rows = driver.find_elements(By.CSS_SELECTOR, ".player-futures-table__player a")

player_urls = []
seen_names = set()

for link in player_rows:
    name = link.text.strip()
    href = link.get_attribute("href")
    if name and name not in seen_names:
        seen_names.add(name)
        player_urls.append((name, href))
    if len(player_urls) >= 100:
        break

print(f"Collected {len(player_urls)} player URLs.")

# Collect data per player
all_data = []

for name, url in player_urls:
    driver.get(url)
    
    WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".player-futures-table__player a"))
)

    try:
        position = driver.find_element(By.CSS_SELECTOR, ".player-info__meta").text.split("•")[0].strip()
    except:
        position = "Unknown"

    rows = driver.find_elements(By.CSS_SELECTOR, ".odds-prop__item")
    for row in rows:
        try:
            category = row.find_element(By.CLASS_NAME, "odds-prop__label").text.strip()
            line = row.find_element(By.CLASS_NAME, "odds-prop__line").text.strip()
            odds = row.find_element(By.CLASS_NAME, "odds-prop__odds").text.strip()
            all_data.append({
                "Player": name,
                "Position": position,
                "Category": category,
                "Line": line,
                "Odds": odds,
                "URL": url
            })
        except:
            continue

# Save to CSV
df = pd.DataFrame(all_data)
df.to_csv("player_futures_odds.csv", index=False)
print("Saved player_futures_odds.csv")

driver.quit()


Collected 0 player URLs.
Saved player_futures_odds.csv


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Set up Chrome options
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--ignore-certificate-errors")

# Start the browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load the player futures page
url = "https://www.bettingpros.com/nfl/odds/player-futures/"
driver.get(url)

# Wait until at least one player card link appears
try:
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.player-card__link"))
    )
except:
    print("❌ Player links didn’t load in time.")
    
# Grab up to 10 player links (you can raise this to 100 later)
link_elements = driver.find_elements(By.CSS_SELECTOR, "a.player-card__link")
player_links = [link.get_attribute("href") for link in link_elements[:30]]  # First 30

# --- Step 2: Visit each player and extract just 'Passing Yards' ---
data = []

for link in player_links:
    full_url = f"https://www.bettingpros.com{link}" if link.startswith("/") else link
    driver.get(full_url)
    time.sleep(3)
    
    # Extract name from URL or header
    player_name = full_url.rstrip("/").split("/")[-1].replace("-", " ").title()
    prop_rows = driver.find_elements(By.CSS_SELECTOR, ".odds-prop__item")

    for row in prop_rows:
        try:
            category = row.find_element(By.CLASS_NAME, "odds-prop__label").text.strip()
            if "Passing Yards" in category:
                line = row.find_element(By.CLASS_NAME, "odds-prop__line").text.strip()
                odds = row.find_element(By.CLASS_NAME, "odds-prop__odds").text.strip()
                data.append({
                    "Player": player_name,
                    "Category": category,
                    "Line": line,
                    "Odds": odds,
                    "URL": full_url
                })
        except Exception as e:
            print(f"Error extracting data for {player_name}: {e}")

df = pd.DataFrame(data)
df.to_csv("passing_yards_props.csv", index=False)
print("✅ Saved passing_yards_props.csv with", len(df), "rows")

driver.quit()


✅ Saved passing_yards_props.csv with 0 rows


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Set up Chrome options
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--ignore-certificate-errors")

# Start the browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load the player futures page
url = "https://www.bettingpros.com/nfl/odds/player-futures/"
driver.get(url)

# Wait until at least one player card link appears
try:
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.player-card__link"))
    )
except:
    print("❌ Player links didn’t load in time.")
    
# Grab up to 10 player links (you can raise this to 100 later)
link_elements = driver.find_elements(By.CSS_SELECTOR, "a.player-card__link")
player_links = [link.get_attribute("href") for link in link_elements[:30]]  # First 30

# --- Step 2: Visit each player and extract just 'Passing Yards' ---
data = []

for link in player_links:
 # --- Debug: Test just the first player ---
    test_url = player_links[0]
    full_url = f"https://www.bettingpros.com{test_url}" if test_url.startswith("/") else test_url
    driver.get(full_url)
    time.sleep(3)

    # Show where we are
    print(f"\n🔍 Visiting: {full_url}")

    # Extract name
    player_name = full_url.rstrip("/").split("/")[-1].replace("-", " ").title()
    print(f"Player Name: {player_name}")

    # Try to find all prop rows
    prop_rows = driver.find_elements(By.CSS_SELECTOR, ".odds-prop__item")
    print(f"Found {len(prop_rows)} total prop blocks")

    # Print each row’s label and line (just for inspection)
    for i, row in enumerate(prop_rows):
        try:
            label = row.find_element(By.CLASS_NAME, "odds-prop__label").text.strip()
            line = row.find_element(By.CLASS_NAME, "odds-prop__line").text.strip()
            odds = row.find_element(By.CLASS_NAME, "odds-prop__odds").text.strip()
            print(f"{i+1}. {label} → {line} ({odds})")
        except Exception as e:
            print(f"{i+1}. ⚠️ Error in row: {e}")


"""df = pd.DataFrame(data)
df.to_csv("passing_yards_props.csv", index=False)
print("✅ Saved passing_yards_props.csv with", len(df), "rows")"""

driver.quit()



🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/playe

Nothing came back. Also it only visited Jayden Daniels

Possible Problems:
* THe time.sleep is not long enough
* Lazy Loading

Possible solutions
* Add code that will scroll to the bottom of the page and then wait 5 seconds to give the page time to load.

In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Set up Chrome options
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--ignore-certificate-errors")

# Start the browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load the player futures page
url = "https://www.bettingpros.com/nfl/odds/player-futures/"
driver.get(url)

# Wait until at least one player card link appears
try:
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.player-card__link"))
    )
except:
    print("❌ Player links didn’t load in time.")
    
# Grab up to 10 player links (you can raise this to 100 later)
link_elements = driver.find_elements(By.CSS_SELECTOR, "a.player-card__link")
player_links = [link.get_attribute("href") for link in link_elements[:5]]  # First 30

# --- Step 2: Visit each player and extract just 'Passing Yards' ---
data = []

for link in player_links:
 # --- Debug: Test just the first player ---
    test_url = player_links[0]
    full_url = f"https://www.bettingpros.com{test_url}" if test_url.startswith("/") else test_url
    driver.get(full_url)
    
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)  

    # Show where we are
    print(f"\n🔍 Visiting: {full_url}")

    # Extract name
    player_name = full_url.rstrip("/").split("/")[-1].replace("-", " ").title()
    print(f"Player Name: {player_name}")

    # Try to find all prop rows
    prop_rows = driver.find_elements(By.CSS_SELECTOR, ".odds-prop__item")
    print(f"Found {len(prop_rows)} total prop blocks")

    # Print each row’s label and line (just for inspection)
    for i, row in enumerate(prop_rows):
        try:
            label = row.find_element(By.CLASS_NAME, "odds-prop__label").text.strip()
            line = row.find_element(By.CLASS_NAME, "odds-prop__line").text.strip()
            odds = row.find_element(By.CLASS_NAME, "odds-prop__odds").text.strip()
            print(f"{i+1}. {label} → {line} ({odds})")
        except Exception as e:
            print(f"{i+1}. ⚠️ Error in row: {e}")


"""df = pd.DataFrame(data)
df.to_csv("passing_yards_props.csv", index=False)
print("✅ Saved passing_yards_props.csv with", len(df), "rows")"""

driver.quit()



🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks


Again no prop blocks. Trynig a new scroll method

In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Set up Chrome options
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--ignore-certificate-errors")

# Start the browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load the player futures page
url = "https://www.bettingpros.com/nfl/odds/player-futures/"
driver.get(url)

# Wait until at least one player card link appears
try:
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.player-card__link"))
    )
except:
    print("❌ Player links didn’t load in time.")
    
# Grab up to 10 player links (you can raise this to 100 later)
link_elements = driver.find_elements(By.CSS_SELECTOR, "a.player-card__link")
player_links = [link.get_attribute("href") for link in link_elements[:5]]  # First 30

# --- Step 2: Visit each player and extract just 'Passing Yards' ---
data = []

for link in player_links:
 # --- Debug: Test just the first player ---
    test_url = player_links[0]
    full_url = f"https://www.bettingpros.com{test_url}" if test_url.startswith("/") else test_url
    driver.get(full_url)
    
    for y in range(0, 1500, 300):  # simulate scrolling every 300px
        driver.execute_script(f"window.scrollTo(0, {y});")
        time.sleep(0.3)

    # Show where we are
    print(f"\n🔍 Visiting: {full_url}")

    # Extract name
    player_name = full_url.rstrip("/").split("/")[-1].replace("-", " ").title()
    print(f"Player Name: {player_name}")

    # Try to find all prop rows
    prop_rows = driver.find_elements(By.CSS_SELECTOR, ".odds-prop__item")
    print(f"Found {len(prop_rows)} total prop blocks")

    # Print each row’s label and line (just for inspection)
    for i, row in enumerate(prop_rows):
        try:
            label = row.find_element(By.CLASS_NAME, "odds-prop__label").text.strip()
            line = row.find_element(By.CLASS_NAME, "odds-prop__line").text.strip()
            odds = row.find_element(By.CLASS_NAME, "odds-prop__odds").text.strip()
            print(f"{i+1}. {label} → {line} ({odds})")
        except Exception as e:
            print(f"{i+1}. ⚠️ Error in row: {e}")


"""df = pd.DataFrame(data)
df.to_csv("passing_yards_props.csv", index=False)
print("✅ Saved passing_yards_props.csv with", len(df), "rows")"""

driver.quit()



🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks

🔍 Visiting: https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
Player Name: Jayden Daniels
Found 0 total prop blocks


Only returning Jayden Daniels for some reason, so lets look at the player links and see why

In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Set up Chrome options
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--ignore-certificate-errors")

# Start the browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load the player futures page
url = "https://www.bettingpros.com/nfl/odds/player-futures/"
driver.get(url)

# Wait until at least one player card link appears
try:
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.player-card__link"))
    )
except:
    print("❌ Player links didn’t load in time.")
    
# Grab up to 10 player links (you can raise this to 100 later)
link_elements = driver.find_elements(By.CSS_SELECTOR, "a.player-card__link")
print(f"Found {len(link_elements)} player links.")
print (f"First 5 links: {[link.get_attribute('href') for link in link_elements[:5]]}")
player_links = [link.get_attribute("href") for link in link_elements]
print(player_links)

for link in player_links[:10]:  # Limit to first 10 for testing
# --- Debug: Test just the first player ---
    driver.get(link)
    
    for y in range(0, 1500, 300):  # simulate scrolling every 300px
        driver.execute_script(f"window.scrollTo(0, {y});")
        time.sleep(0.3)

    # Extract name
    player_name = link.rstrip("/").split("/")[-1].replace("-", " ").title()
    print(f"Player Name: {player_name}")

    # Try to find all prop rows
    prop_rows = driver.find_elements(By.CSS_SELECTOR, ".odds-prop__item")
    print(f"Found {len(prop_rows)} total prop blocks")

    # Print each row’s label and line (just for inspection)
    for i, row in enumerate(prop_rows):
        try:
            label = row.find_element(By.CLASS_NAME, "typography odds-market-label").text.strip()
            line = row.find_element(By.CLASS_NAME, "typography odds-cell_cost").text.strip()
            odds = row.find_element(By.CLASS_NAME, "typography odds-cell_line").text.strip()
            print(f"{i+1}. {label} → {line} ({odds})")
        except Exception as e:
            print(f"{i+1}. ⚠️ Error in row: {e}")

Found 687 player links.
First 5 links: ['https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/', 'https://www.bettingpros.com/nfl/odds/player-futures/jalen-hurts/', 'https://www.bettingpros.com/nfl/odds/player-futures/josh-allen-qb/', 'https://www.bettingpros.com/nfl/odds/player-futures/justin-fields/', 'https://www.bettingpros.com/nfl/odds/player-futures/travis-hunter/']
['https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/', 'https://www.bettingpros.com/nfl/odds/player-futures/jalen-hurts/', 'https://www.bettingpros.com/nfl/odds/player-futures/josh-allen-qb/', 'https://www.bettingpros.com/nfl/odds/player-futures/justin-fields/', 'https://www.bettingpros.com/nfl/odds/player-futures/travis-hunter/', 'https://www.bettingpros.com/nfl/odds/player-futures/lamar-jackson/', 'https://www.bettingpros.com/nfl/odds/player-futures/kyler-murray/', 'https://www.bettingpros.com/nfl/odds/player-futures/cooper-kupp/', 'https://www.bettingpros.com/nfl/odds/player-futures/j

Ok so we changed now to having multiple players by getting rid of test_url = player_links[0], but still getting 0 total prop blocks per player.

Posible solution, remove the typography from the CLASS_NAME, because thats how it works lol. That class doesn't exist.

In [2]:
# This works for grabbing the player links from bettingpros.com
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Set up Chrome options
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--ignore-certificate-errors")

# Start the browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load the player futures page
url = "https://www.bettingpros.com/nfl/odds/player-futures/"
driver.get(url)

# Wait until at least one player card link appears
try:
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.player-card__link"))
    )
except:
    print("❌ Player links didn’t load in time.")
    
# Grab up to 10 player links (you can raise this to 100 later)
link_elements = driver.find_elements(By.CSS_SELECTOR, "a.player-card__link")
print(f"Found {len(link_elements)} player links.")
print (f"First 5 links: {[link.get_attribute('href') for link in link_elements[:5]]}")
player_links = [link.get_attribute("href") for link in link_elements]


Found 687 player links.
First 5 links: ['https://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/', 'https://www.bettingpros.com/nfl/odds/player-futures/jalen-hurts/', 'https://www.bettingpros.com/nfl/odds/player-futures/josh-allen-qb/', 'https://www.bettingpros.com/nfl/odds/player-futures/justin-fields/', 'https://www.bettingpros.com/nfl/odds/player-futures/travis-hunter/']


In [4]:
import time

for link in player_links[:1]:  # Limit to first 10 for testing
# --- Debug: Test just the first player ---
    driver.get(link)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(15)  # let JS load    
    
    #Attempt 1
    #################################################
    
    """    # Wait for the number of prop blocks to increase (e.g., at least 10)
    WebDriverWait(driver, 15).until(
        lambda d: len(d.find_elements(By.CLASS_NAME, "grouped-items-with-sticky-footer__content")) > 5
    )

    time.sleep(2)  # Optional: add a little extra wait for safety"""
    #Attempt 2
    #################################################
    """    last_count = 0
    for _ in range(10):  # Try up to 10 times
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(5)
        prop_rows = driver.find_elements(By.CLASS_NAME, "grouped-items-with-sticky-footer__content")
        if len(prop_rows) == last_count:
            break  # No new elements loaded, we're done
        last_count = len(prop_rows)"""
    #Attempt 3    
    #################################################
    
    last_count = 0
    stable_count = 0
    max_wait = 15  # seconds
    start = time.time()

    while time.time() - start < max_wait:
        prop_rows = driver.find_elements(By.CLASS_NAME, "grouped-items-with-sticky-footer__content")
        if len(prop_rows) == last_count:
            stable_count += 1
            if stable_count >= 3:  # Number hasn't changed for 3 checks (~1.5s)
                break
        else:
            stable_count = 0
            last_count = len(prop_rows)
        time.sleep(0.5)
    
    print(f"Found {len(prop_rows)} total prop blocks")
    
    # Extract name
    player_name = link.rstrip("/").split("/")[-1].replace("-", " ").title()
    print(f"Player Name: {player_name}")

    # Try to find all prop rows
    prop_rows = driver.find_elements(By.CLASS_NAME, "grouped-items-with-sticky-footer__content")
    print(f"Found {len(prop_rows)} total prop blocks")


Found 5 total prop blocks
Player Name: Jayden Daniels
Found 5 total prop blocks


In [ ]:

    # Print each row’s label and line (just for inspection)
    for i, row in enumerate(prop_rows):
        try:
            # Extract the prop label (e.g., "Total Passing Yards")
            #label = row.find_element(By.CLASS_NAME, "odds-market-label").text.strip()

            #if "Passing Yards" in label:
            line = row.find_elements(By.XPATH, ".//span[contains(@class, 'odds-cell__line')]")[1].text.strip()
                #odds = row.find_element(By.CLASS_NAME, "odds-cell__cost").text.strip()
            print(f"{i+1}. {line} ")
        except Exception as e:
            print(f"{i+1}. ⚠️ Skipped row:", e)

Still nothing. Chaning the code to remove 